In [1]:
import pandas as pd
import numpy as np
import pickle

from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import root_mean_squared_error , mean_absolute_error , mean_absolute_percentage_error          

pd.options.display.float_format = '{:,.2f}'.format
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

## 1. Data Preparation

In [2]:
path = '../datasets/'

df = pd.read_csv(f"{path}/dataset_training.csv")


In [3]:
def data_preparation(df):
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')

    X_columns = df.drop(['unemployment_rate','unemployment_rate_diff'], axis = 1)

    selected_cols = ['month', 
                'employment_rate_diff', 
                'industry_production_prices_diff',
                'companies_trust',  
                'rolling_mean_6m',
                'rolling_mean_12m', 
                'PCA_labour_market_1',
                'PCA_labour_market_2', 
                'PCA_price_energy_1', 
                'PCA_price_energy_2'
                ]

    exog_train = X_columns.iloc[5:-12][selected_cols]
    exog_test = X_columns.iloc[-12:][selected_cols]

    y_train = df.iloc[5:-12]['unemployment_rate_diff']
    y_test = df.iloc[-12:]['unemployment_rate_diff']

    return exog_train, exog_test, y_train, y_test


In [22]:
# exog_train, exog_test, y_train, y_test = data_preparation(df)

## 2. Model: SARIMAX

In [23]:
# # best_param = (1, 1, 0)
# # best_param_seasonal = (0, 1, 0, 12)

# best_param = (0, 0, 0)
# best_param_seasonal = (1, 0, 0, 12)

In [4]:
def model(df):
    exog_train, exog_test, y_train, y_test = data_preparation(df)
    
    best_param = (0, 0, 0)
    best_param_seasonal = (1, 0, 0, 12)
    
    model = SARIMAX(y_train,
                    exog=exog_train, 
                    order=best_param,
                    seasonal_order=best_param_seasonal,
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    results = model.fit()

    y_test_pred = results.predict(
                            start=exog_test.index[0],
                            end=exog_test.index[-1],
                            exog=exog_test
                                )

    test_rmse = round(root_mean_squared_error(y_test, y_test_pred), 5 )
    print(f"RMSE: {test_rmse}")

    anchor =  df['unemployment_rate'].iloc[-13] 

    df_prection = pd.DataFrame({
        "date":y_test_pred.index,
        "unemployment_rate_fcst":anchor + y_test_pred.cumsum().values
        })

    df_final = df.merge(df_prection, on= 'date', how = 'left')
    
    return  df_final , results   

In [7]:
df_final, results = model(df)

RMSE: 0.02812


In [9]:
selected_cols = ['month', 
                'employment_rate_diff', 
                'industry_production_prices_diff',
                'companies_trust',  
                'rolling_mean_6m',
                'rolling_mean_12m', 
                'PCA_labour_market_1',
                'PCA_labour_market_2', 
                'PCA_price_energy_1', 
                'PCA_price_energy_2'
                ]

with open('../models/sarimax_unemployment_12m.pkl', 'wb') as f:
    pickle.dump({'model': results, 'selected_cols': selected_cols}, f)

In [10]:
df_final.to_csv(f"{path}dataset_predicion.csv")